# Alexinator : entraîner notre propre gardien (QLoRA)

**But.** Spécialiser un LLM open source pour qu'il tienne le rôle du gardien : donner des indices, refuser les manipulations, ne jamais trahir le film. On compare ensuite le modèle de base et le modèle entraîné.

**Méthode.** On n'entraîne pas un LLM à partir de zéro (il faudrait des milliers d'heures de GPU). On fait un *affinage* (fine-tuning) :

- le modèle de base (environ 7 milliards de paramètres) est chargé en **4 bits** : il occupe environ 6 Go au lieu de 15 ;
- ses poids sont **gelés**. On ajoute de petites matrices entraînables, les adaptateurs **LoRA**, qui représentent moins de 1 % des paramètres ;
- la boucle d'entraînement est celle des labs : `forward → loss → backward → step`.

L'ensemble tient dans les 24 Go de la carte graphique.

**Avant de lancer ce notebook**, il faut avoir fabriqué les données avec `python finetune/construire_dataset.py` (fichiers `finetune/data/train.jsonl` et `val.jsonl`).

> Ce notebook n'a pas pu être exécuté lors de sa rédaction (pas de GPU disponible). Lancez les cellules une par une la première fois, et commencez par un essai court (`EPOCHS = 1`, quelques centaines d'exemples).

## 0. Installation

À lancer une fois sur le serveur, puis redémarrer le noyau.

In [ ]:
# %pip install -U torch transformers peft bitsandbytes accelerate matplotlib pyyaml

## 1. Réglages

Tous les paramètres à ajuster sont ici.

In [ ]:
import json
import math
import random
import sys
import time
from pathlib import Path

# Racine du projet (le notebook peut être lancé depuis finetune/ ou depuis la racine)
ROOT = Path.cwd() if (Path.cwd() / "alexinator").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

# --- Modèle de base -------------------------------------------------------------------
# Un modèle « instruct » de 7 à 8 milliards de paramètres, multilingue, qui accepte un message
# système. Qwen2.5-7B-Instruct est sous licence Apache 2.0 et se télécharge sans demande d'accès.
# Pour en essayer un autre, changez seulement cette ligne.
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# --- Données et sorties ---------------------------------------------------------------
TRAIN_PATH = ROOT / "finetune" / "data" / "train.jsonl"
VAL_PATH = ROOT / "finetune" / "data" / "val.jsonl"
OUTPUT_DIR = ROOT / "finetune" / "gardien-lora"      # l'adaptateur entraîné sera enregistré ici

# --- Hyperparamètres ------------------------------------------------------------------
MAX_LENGTH = 1024       # longueur maximale d'un exemple, en tokens
EPOCHS = 3              # nombre de passages sur les données
BATCH_SIZE = 4          # exemples traités en même temps (à baisser si « CUDA out of memory »)
GRAD_ACCUM = 4          # on cumule 4 batchs avant chaque mise à jour : batch effectif = 16
LEARNING_RATE = 2e-4    # valeur courante pour LoRA
WARMUP_RATIO = 0.05     # montée progressive du learning rate au début
LORA_R = 16             # rang des adaptateurs : plus grand = plus de capacité (et de mémoire)
LORA_ALPHA = 32         # facteur d'échelle (souvent 2 x r)
LORA_DROPOUT = 0.05
SEED = 0

random.seed(SEED)

In [ ]:
import torch

torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Aucun GPU détecté : ce notebook doit tourner sur le serveur."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}, {gpu.total_memory / 1e9:.0f} Go")

## 2. Les données

Chaque ligne est une conversation `[system, user, assistant]` :

- `system` : le prompt du gardien avec la fiche du film ;
- `user` : un message de joueur (attaque ou question honnête) ;
- `assistant` : la réponse idéale, écrite par le modèle professeur et validée par nos défenses.

Les films de validation n'apparaissent pas dans l'entraînement.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_examples = read_jsonl(TRAIN_PATH)
val_examples = read_jsonl(VAL_PATH)
print(f"{len(train_examples)} exemples d'entraînement, {len(val_examples)} de validation")
print("Films de validation :", sorted({e["film_id"] for e in val_examples}))
assert not {e["film_id"] for e in train_examples} & {e["film_id"] for e in val_examples}, "fuite train/val !"

exemple = random.choice(train_examples)
print("\n--- Exemple ---")
print("JOUEUR  :", exemple["messages"][1]["content"])
print("GARDIEN :", exemple["messages"][2]["content"])

## 3. Le modèle : chargement en 4 bits et adaptateurs LoRA

- `BitsAndBytesConfig(load_in_4bit=True)` : chaque poids est stocké sur 4 bits au lieu de 16 (c'est le « Q » de QLoRA).
- `prepare_model_for_kbit_training` : gèle le modèle et active le *gradient checkpointing* (moins de mémoire, un peu plus lent).
- `LoraConfig` : pour chaque couche linéaire visée, de poids $W$, on apprend deux petites matrices $A$ et $B$ de rang `r`, et la couche calcule $Wx + \frac{\alpha}{r} BAx$. Seules $A$ et $B$ reçoivent des gradients.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:            # certains modèles n'ont pas de token de remplissage
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,   # les calculs, eux, se font en 16 bits
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quantization, device_map={"": 0})
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False               # inutile pendant l'entraînement

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    # Couches d'attention et couches MLP de chaque bloc du transformer.
    # Ces noms conviennent à Qwen, Llama et Mistral ; pour un autre modèle, vérifier avec print(model).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

## 4. Préparer les exemples : on n'apprend que la réponse du gardien

Le *chat template* du modèle transforme la conversation en un seul texte avec ses balises spéciales. On le coupe en deux :

- le **contexte** (système + joueur + balise « à toi de répondre ») : le modèle le lit, mais on ne lui demande pas de le prédire. Ses étiquettes valent `-100`, valeur ignorée par la cross-entropy ;
- la **réponse** du gardien et son marqueur de fin : c'est elle que le modèle apprend à produire.

Sans ce masque, le modèle gaspillerait sa capacité à réciter le prompt système.

In [ ]:
def encode(example):
    messages = example["messages"]
    # Texte du contexte, terminé par la balise qui annonce la réponse de l'assistant
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    # Texte complet : contexte + réponse + marqueur de fin de tour
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    if full_text.startswith(prompt_text):
        answer_text = full_text[len(prompt_text):]
    else:  # template inhabituel : on construit la fin à la main
        answer_text = messages[-1]["content"] + tokenizer.eos_token

    # add_special_tokens=False : le template a déjà placé les tokens spéciaux
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_LENGTH]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LENGTH]
    return {"input_ids": input_ids, "labels": labels}


def encode_all(examples):
    encoded = [encode(e) for e in examples]
    kept = [e for e in encoded if any(label != -100 for label in e["labels"])]
    if len(kept) < len(encoded):
        print(f"{len(encoded) - len(kept)} exemple(s) trop long(s) écarté(s) : augmenter MAX_LENGTH")
    return kept


train_data = encode_all(train_examples)
val_data = encode_all(val_examples)

# Vérification visuelle : ce que le modèle lit, et ce qu'il doit apprendre à écrire
sample = train_data[0]
learned = [t for t, label in zip(sample["input_ids"], sample["labels"]) if label != -100]
print("Longueur moyenne :", sum(len(e["input_ids"]) for e in train_data) // len(train_data), "tokens")
print("\n--- Fin du contexte (non appris) ---\n", tokenizer.decode(sample["input_ids"][:len(sample["input_ids"]) - len(learned)])[-300:])
print("\n--- Partie apprise ---\n", tokenizer.decode(learned))

In [ ]:
from torch.utils.data import DataLoader


def collate(batch):
    """Aligne les exemples d'un batch sur le plus long, en complétant à droite."""
    longest = max(len(e["input_ids"]) for e in batch)
    input_ids, labels, attention_mask = [], [], []
    for e in batch:
        pad = longest - len(e["input_ids"])
        input_ids.append(e["input_ids"] + [tokenizer.pad_token_id] * pad)
        labels.append(e["labels"] + [-100] * pad)                 # le remplissage n'est pas appris
        attention_mask.append([1] * len(e["input_ids"]) + [0] * pad)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attention_mask)}


train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
print(f"{len(train_loader)} batchs d'entraînement par époque")

## 5. Mesure avant entraînement

On mesure le modèle de base pour avoir un point de comparaison :

- la **loss de validation** ;
- le **taux de fuite** : sur des attaques visant les films de validation, part des réponses qui contiennent le titre ou un nom de la fiche. On réutilise le filtre de sortie du jeu (`defenses.output_filter`).

In [ ]:
from alexinator import defenses
from alexinator.films import load_films

FILMS = {f.id: f for f in load_films()}


@torch.no_grad()
def generate(messages, max_new_tokens=150):
    """Réponse du modèle à une conversation (génération déterministe, pour des mesures reproductibles)."""
    model.eval()
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                           return_dict=True).to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.pad_token_id)
    model.train()
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


@torch.no_grad()
def validation_loss():
    model.eval()
    losses = []
    for batch in val_loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            losses.append(model(**batch).loss.item())
    model.train()
    return sum(losses) / len(losses)


# Toujours les mêmes attaques, pour comparer avant et après
eval_attacks = [e for e in val_examples if e["type"].startswith("attaque")]
eval_attacks = random.Random(SEED).sample(eval_attacks, min(40, len(eval_attacks)))


def leak_rate(show=0):
    leaks = 0
    for i, e in enumerate(eval_attacks):
        reply = generate(e["messages"][:2])                       # système + message du joueur
        leak = defenses.output_filter(reply, FILMS[e["film_id"]]).blocked
        leaks += leak
        if i < show:
            print(f"JOUEUR  : {e['messages'][1]['content']}\nGARDIEN : {reply}\n-> {'FUITE' if leak else 'ok'}\n")
    return leaks / max(1, len(eval_attacks))


loss_before = validation_loss()
leak_before = leak_rate(show=3)
print(f"Avant entraînement : loss de validation = {loss_before:.3f}, taux de fuite = {leak_before:.0%}")

## 6. La boucle d'entraînement

La même que dans les labs, avec trois ajouts utiles pour un gros modèle :

- **accumulation de gradients** : on additionne les gradients de `GRAD_ACCUM` petits batchs avant de faire un pas, ce qui simule un grand batch sans la mémoire correspondante ;
- **`torch.autocast`** : les calculs se font en 16 bits (plus rapide, moins de mémoire) ;
- **planificateur** : le learning rate monte doucement (*warmup*) puis redescend en cosinus.

À la fin de chaque époque, on mesure la loss de validation et on garde les meilleurs adaptateurs. Si la loss de validation remonte alors que celle d'entraînement baisse, le modèle sur-apprend : c'est le signal pour s'arrêter.

In [ ]:
from transformers import get_cosine_schedule_with_warmup

trainable = [p for p in model.parameters() if p.requires_grad]      # uniquement les matrices LoRA
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)

train_curve, val_curve = [], [(0, loss_before)]
best_val, best_state = float("inf"), None
step_count, start = 0, time.time()

model.train()
optimizer.zero_grad()
for epoch in range(EPOCHS):
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}

        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss               # forward : cross-entropy sur la réponse du gardien
        (loss / GRAD_ACCUM).backward()               # backward : les gradients s'accumulent

        if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(train_loader):
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)   # évite les pas trop grands
            optimizer.step()                                 # mise à jour des adaptateurs
            scheduler.step()
            optimizer.zero_grad()
            step_count += 1
            train_curve.append((step_count, loss.item()))
            if step_count % 10 == 0:
                print(f"époque {epoch + 1}, pas {step_count}/{total_steps}, loss = {loss.item():.3f}, "
                      f"{(time.time() - start) / 60:.1f} min")

    val = validation_loss()
    val_curve.append((step_count, val))
    print(f"=== Fin de l'époque {epoch + 1} : loss de validation = {val:.3f} ===")
    if val < best_val:   # on garde une copie des meilleurs adaptateurs
        best_val = val
        best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

# On remet dans le modèle les adaptateurs de la meilleure époque
with torch.no_grad():
    for n, p in model.named_parameters():
        if p.requires_grad and best_state is not None:
            p.copy_(best_state[n].to(p.device))
print(f"Terminé en {(time.time() - start) / 60:.1f} min. Meilleure loss de validation : {best_val:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(*zip(*train_curve), alpha=0.5, label="entraînement (par pas)")
plt.plot(*zip(*val_curve), marker="o", label="validation (par époque)")
plt.xlabel("pas d'optimisation")
plt.ylabel("loss (cross-entropy)")
plt.title("Courbes d'apprentissage du gardien")
plt.legend()
plt.show()

## 7. Mesure après entraînement

Mêmes attaques, mêmes films de validation (jamais vus à l'entraînement).

In [ ]:
loss_after = validation_loss()
leak_after = leak_rate(show=3)
print(f"Loss de validation : {loss_before:.3f} -> {loss_after:.3f}")
print(f"Taux de fuite      : {leak_before:.0%} -> {leak_after:.0%}")

Deux précautions pour interpréter ces chiffres :

- le taux de fuite ne compte que les fuites **visibles** par le filtre (titre, noms). La mesure complète, avec le devineur et les questions honnêtes, se fait avec `campagne.py` (voir plus bas) ;
- un modèle qui refuse tout a un taux de fuite nul mais rend le jeu injouable. Vérifiez à la main qu'il donne encore des indices :

In [ ]:
honest = [e for e in val_examples if e["type"] == "honnete"][:5]
for e in honest:
    print(f"JOUEUR  : {e['messages'][1]['content']}\nGARDIEN : {generate(e['messages'][:2])}\n")

## 8. Enregistrer le modèle

On n'enregistre que les adaptateurs LoRA (quelques dizaines de Mo), pas le modèle de base.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(OUTPUT_DIR / "resultats.json", "w", encoding="utf-8") as f:
    json.dump({"base_model": BASE_MODEL, "epochs": EPOCHS, "lora_r": LORA_R, "learning_rate": LEARNING_RATE,
               "train_examples": len(train_data), "loss_before": loss_before, "loss_after": loss_after,
               "leak_before": leak_before, "leak_after": leak_after}, f, indent=2)
print("Adaptateur enregistré dans", OUTPUT_DIR)

## 9. Jouer avec notre modèle et l'évaluer

Dans le fichier `.env` à la racine du projet (sur le serveur) :

```
ALEXINATOR_BACKEND=local
ALEXINATOR_MODEL=Qwen/Qwen2.5-7B-Instruct
ALEXINATOR_ADAPTER=finetune/gardien-lora
```

Puis, depuis la racine du projet :

```
python cli.py --niveau 2                                          jouer contre notre gardien
python campagne.py --niveaux 2 --films 4 --nom modele_entraine    campagne d'attaques complète
```

Pour la comparaison du rapport, lancez la même campagne avec le modèle de base (retirer la ligne `ALEXINATOR_ADAPTER`, `--nom modele_de_base`), puis ouvrez `analyse.ipynb`, section B.3.

Avec `ALEXINATOR_BACKEND=local`, le gardien et les défenses (censeur, détecteur, devineur) utilisent le même modèle chargé sur le GPU.

## Pour aller plus loin

- Plus de données : davantage de fiches de films, les tentatives réelles des playtests (`--journal`), des conversations à plusieurs tours.
- Comparer plusieurs rangs LoRA (`LORA_R` = 8, 16, 64) ou plusieurs modèles de base.
- Entraîner aussi le détecteur d'attaques : un petit modèle classifieur sur les messages étiquetés du journal.